In [30]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from rich.progress import track
import urllib.request
import math

In [25]:
class Embedding(nn.Module):
  def __init__(self, dims: int=64, vocab_size: int=5000, dropout: float=0.3):
    super().__init__()

    self.embed = nn.Embedding(vocab_size, dims)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    return self.dropout(self.embed(x))

In [63]:
class PositionalEmbedding(nn.Module):
  def __init__(self, dims: int=64, vocab_size: int=5000):
    super().__init__()

    pe = torch.zeros(vocab_size, dims)

    position = torch.arange(0, vocab_size, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, dims, 2).float() * (-math.log(10000.0) / dims))

    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)

    self.pe = pe.unsqueeze(0)

  def forward(self, x):
    x = x + self.pe[:, :x.size(1)]
    return x

In [64]:
class Attention(nn.Module):
  def __init__(self, dims: int=64, dropout: float=0.3):
    super().__init__()

    self.dims = dims
    self.dropout = nn.Dropout(dropout)

    self.q = nn.Linear(dims, dims)
    self.k = nn.Linear(dims, dims)
    self.v = nn.Linear(dims, dims)

  def forward(self,x):
    q = self.q(x)
    k = self.k(x)
    v = self.v(x)

    attention = torch.matmul(q, k.transpose(-2, -1)) / (self.dims ** 0.5)
    return attention #later soft max

In [72]:
class PostAttention(nn.Module):
  def __init__(self, dims: int=64, dropout: float=0.3):
    super().__init__()

    self.norm1 = nn.LayerNorm(dims)
    self.drop1 = nn.Dropout(dropout)

    self.fnn = nn.Sequential(
        nn.Linear(dims, dims*4),
        nn.ReLU(),
        nn.Linear(dims*4, dims)
    )

    self.norm2 = nn.LayerNorm(dims)
    self.drop2 = nn.Dropout(dropout)

  def forward(self, x, attention):
    x = x + self.drop1(torch.matmul(attention, x))
    x = self.norm1(x)

    x = self.fnn(x)

    x = x + self.drop2(self.fnn(x))
    x = self.norm2(x)

    return x

In [74]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, "text.txt")
with open("text.txt") as f:
    text = f.read(5000)

words = text.split()

vocab = sorted(list(set(words)))

wti = {word: i for i, word in enumerate(vocab)}
itw = {i: word for i, word in enumerate(vocab)}

seq = 5

X = []
y = []

for i in range(len(words)-seq):
  X.append([wti[w] for w in words[i:i+seq]])
  y.append([wti[words[i+seq]]])

X = torch.tensor(X)
y = torch.tensor(y)

x_test = torch.tensor([[21,  15,  10, 450]])
y_test = torch.tensor([[332]])

embeddings = Embedding(vocab_size=len(vocab))
pos_embeddings = PositionalEmbedding(vocab_size=len(vocab))
attention = Attention()

word_embedding = embeddings(x_test)
input = pos_embeddings(word_embedding)
encoder_output = PostAttention(dropout=0.3)

attention_output = attention(input)
encoder_output(input, attention_output)

tensor([[[-0.6994, -1.9009, -0.2767,  2.3805,  1.3809, -1.4683, -0.0081,
           0.1391, -2.0772, -0.3353, -0.0101, -0.4139, -0.6594,  0.0758,
           1.0022,  1.0270,  0.1429, -1.1648,  1.1426,  1.2014,  0.1033,
          -0.4010,  0.1736,  0.5409,  1.4776,  0.2329,  0.4867, -1.2644,
           0.6033,  0.9443,  0.5996,  1.0509, -0.9235, -0.4269, -1.5993,
           0.0611,  1.9024, -0.2592, -2.0000, -0.2066,  0.3521, -0.3641,
           0.4670,  0.1468,  0.1725,  0.3717, -1.1226,  0.4146,  0.2058,
           0.8291,  0.6677, -0.7981, -0.8789,  1.5137, -0.8306,  0.7073,
           0.0580,  0.5623, -1.9992,  0.8407, -1.6519, -1.7278,  0.4152,
           1.0746],
         [ 0.6482, -1.1446, -0.6142,  0.5857,  2.6092, -1.0757,  0.6745,
          -0.0872, -1.3321, -1.3649, -0.5780,  0.0647,  1.5001,  0.5904,
           0.8475,  0.2084,  0.6373,  0.1164,  1.0896, -0.2137,  1.5562,
           1.4499, -0.1677, -0.0533,  1.7498,  0.1595, -0.2445, -1.6653,
          -1.3401, -1.2845, -1.